# Building the Taxonomy

#### Imports

In [1]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import sys
import os

# Add the parent directory (or another appropriate path) to sys.path so Python can find Exclusion_functions
notebook_dir = os.path.dirname(os.path.abspath('04_Optimization.ipynb'))
parent_dir = os.path.abspath(os.path.join(notebook_dir, '..', '..', '..'))
function_dir = os.path.abspath(os.path.join(parent_dir, 'Function_Files'))
if function_dir not in sys.path:
    sys.path.append(function_dir)

import pandas as pd
from pyvent.tools.llm.openai_api import OpenAIAgent
import nest_asyncio
nest_asyncio.apply()
import Load_Isolate_functions as lif
import Classification_functions as cf
import datetime

#### Variables

In [5]:
CATEGORY = "Cups"
today = datetime.datetime.now().strftime('%m.%d.%Y')
ip_path = f"{parent_dir}\\Data\\"
ip_path_cat = f"{parent_dir}\\Data\\{CATEGORY}\\"
OP_PATH = f"{parent_dir}\\Data\\{CATEGORY}\\Output\\"

#### Read in data
sfy = salsify, to_keep = files received from ID

In [8]:
#Read in files, define variables.
sfy = pd.read_excel(f"{ip_path}All Salsify Items.xlsx", sheet_name='in', skiprows=1)
transactions = pd.read_excel(f"{ip_path_cat}Cups Sales Data L6M Feb-Jul.xlsx")
item_master = pd.read_csv(f"{ip_path}consolidated_item_master_by_location_20250610152008.csv")

C:\Users\zwayne\AppData\Local\Temp\ipykernel_14172\2707040030.py:4: DtypeWarning: Columns (1,2,3,4,5,7,8,9,11,12,13,14,16,17,18,19,20,21,22,24,28,29,30,31,32,33,34,35,36,37,39,40,41,42,43,44,45,46,47,48,51) have mixed types. Specify dtype option on import or set low_memory=False.
  item_master = pd.read_csv(f"{ip_path}consolidated_item_master_by_location_20250610152008.csv")


In [9]:
mapping_dict, erp_system_mapped = lif.map_erp_system_to_number(transactions['ERP System'])

transactions['Entity--Item'] = transactions.apply(
    lambda row: (
        f"{mapping_dict.get(str(row['ERP System']).strip(), 0)}--{str(row['Item']).strip().upper()}"
    ),
    axis=1
)

item_master['Entity--Item'] = item_master.apply(
    lambda row: (
        f"{mapping_dict.get(str(row['erp_system_name']).strip(), 0)}--{str(row['item_code']).strip().upper()}"
    ),
    axis=1
)

sfy['Entity--Item'] = '1--'+sfy['S2K Item Number'].astype(str).str.strip().str.upper()
transactions = transactions[transactions['Item Sub Category'] == 'Cups']

In [10]:
item_master = item_master[item_master['Entity--Item'].isin(transactions['Entity--Item'])]

In [11]:
transactions_added = lif.add_po_cost(transactions, item_master)

  Updated 91 transactions with Qty = Net Cost / PO Cost
  Updated 750 transactions with Gross Cost = PO Cost × Qty
  Updated 742 transactions with Net Cost = Gross Cost (POD = 'N')
  Updated 13846 transactions with po_cost_amt = Gross Cost / Qty
PO Cost matching complete:
  Total transactions: 202321
  Matched transactions: 201481
  Match rate: 99.6%
  Total data fixes applied: 15429 (Qty: 91, Gross Cost: 750, Net Cost: 742, PO Cost: 13846)
  Data cleanup complete:
    Removed 6663 rows with negative values
    Final row count: 195658 (from 202321)


In [12]:
im_grp = lif.group_data(transactions_added)

In [13]:
transactions_added.to_csv(f"{OP_PATH}Cups Sales Data L6M Feb-Jul - Cleaned.csv", index=False)

#### Get S2k Items

In [14]:
im_s2k, columns_with_coverage, example_data = lif.get_columns_with_coverage(im_grp, sfy, 15)

Filtered sfy to 1221 rows
Found 21 columns meeting 15% coverage
Extracted sample data for 21 columns


In [242]:
example_data

,Beverage Cup Type,Product Type Collapse,Pack Size,Color,Material,Foodservice Global Attributes,Usage Temperature,Beverage Cup Style,Sustainable Products,Product Capacity,...,Capacity Preferred Metric 1,"Lid, Cover & Cap Type","Lid, Cover & Cap Style",Compatible Product & Product Type,Product Attributes,Bottom Diameter (IN),Top Diameter (IN),Product Dimension Type,Wall Height (IN),Product Dimensions
0,Cup,Cup,1000/Case,White,Single Wall Poly-Coated Paper,Disposable,Hot Only,Insulated,Yes,16 OZ,...,Ounce (OZ),Dome,Sip Through,Cup,WhiteSingle Wall Poly-Coated PaperDisposableHo...,2.3,3.5,Tapered & Graduated Product Dimensions,5.30,3.5X5.3X2.3 IN
1,Lid,Lid,1080/Case,Black,Polystyrene (PS),Disposable,Hot Only,Insulated,Yes,12-16-20 OZ,...,Ounce (OZ),Flat,With Hole,Cup,BlackPolystyrene (PS)DisposableHot OnlyInsulat...,2.4,11.4,Tapered & Graduated Product Dimensions,2.70,3.7X2.7 IN
2,Cup,Cup,252/Case,White,Polystyrene Foam,Disposable,Cold & Hot,Dessert,Yes,20 OZ,...,Ounce (OZ),Dome,With Hole,Cup,WhitePolystyrene FoamDisposableCold & HotInsul...,1.8,3.7,Tapered & Graduated Product Dimensions,5.50,3.9X5.5X2.4 IN
3,Cup,Cup,500/Case,White,Paper,Disposable,Cold Only,Shot,Yes,44 OZ,...,Ounce (OZ),Dome,Reclosable Tab,Cup,WhiteCoca-Cola|Stock PrintPaperCold Only44 OZ4...,2.4,3.9,Tapered & Graduated Product Dimensions,3.40,4.3X0.4 IN
4,Cup,Cup,1000/Case,Clear,Paper,Disposable,Cold Only,Cone,Yes,16 OZ,...,Ounce (OZ),Flat,Identification,Cup,PaperDisposableCold Only16 OZ16Ounce (OZ),2.1,2.9,Standard Product Dimensions,4.70,2.9X3.4X1.8 IN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,Cup,Cup,1000/Case,White,Plastic,Disposable,Cold Only,Insulated,Yes,8 OZ,...,Ounce (OZ),Dome,Sip Through|Travel,Cup,PlainRoundKraftDouble Wall Poly-Coated PaperHo...,2.5,3.3,Tapered & Graduated Product Dimensions,4.20,3.4X0.7 IN
96,Lid,Cup,1000/Case,Clear,Polyethylene Terephthalate (PET),Disposable,Cold Only,Tumbler,Yes,12-24 OZ,...,Ounce (OZ),Flat,With Hole|Identification,Cup,WhitePaperHot Only8 OZ8Ounce (OZ),1.1,3.5,Tapered & Graduated Product Dimensions,4.50,3.75X5X2.5 IN
97,Cup,Cup Sleeve,1000/Case,White,Oriented Polystyrene (OPS),Disposable,Cold Only,Insulated,Yes,24 OZ,...,Ounce (OZ),Flat,With Hole|Lock Tab|Sip Through,Cup,TranslucentHigh Impact Polystyrene (HIPS)|Orie...,2.9,4.1,Standard Product Dimensions,4.20,3.75X2.75X2.25 IN
98,Lid,Cup,800/Case,White,Polystyrene Foam,Disposable,Cold Only,Tumbler,Yes,8-10 OZ,...,Ounce (OZ),Dome,Strawless,Cup,4Paper8-32 OZ8|32Ounce (OZ),2.4,3.5,Tapered & Graduated Product Dimensions,1.75,4.4X0.4 IN


In [15]:
columns_with_coverage

['Beverage Cup Type',
 'Product Type Collapse',
 'Pack Size',
 'Color',
 'Material',
 'Foodservice Global Attributes',
 'Usage Temperature',
 'Beverage Cup Style',
 'Sustainable Products',
 'Product Capacity',
 'Capacity Value 1',
 'Capacity Preferred Metric 1',
 'Lid, Cover & Cap Type',
 'Lid, Cover & Cap Style',
 'Compatible Product & Product Type',
 'Product Attributes',
 'Bottom Diameter (IN)',
 'Top Diameter (IN)',
 'Product Dimension Type',
 'Wall Height (IN)',
 'Product Dimensions']

In [16]:
# Can manually check/change the columns with coverage to see if they are correct
columns_for_description = ['Beverage Cup Type',
 'Product Type Collapse',
 'Pack Size',
 'Color',
 'Material',
 'Foodservice Global Attributes',
 'Usage Temperature',
 'Beverage Cup Style',
 'Sustainable Products',
 'Product Capacity',
 'Capacity Value 1',
 'Capacity Preferred Metric 1',
 'Lid, Cover & Cap Type',
 'Lid, Cover & Cap Style',
 'Compatible Product & Product Type',
 'Bottom Diameter (IN)',
 'Top Diameter (IN)',
 'Product Dimension Type',
 'Wall Height (IN)',
 'Product Dimensions']

#### Merge data with salsify

In [17]:
# get columns with coverage from sfy dataframe and merge with im_concat dataframe
sfy_covered = sfy[columns_for_description+['Entity--Item']].copy()
im_final = im_grp.merge(sfy_covered, on='Entity--Item', how='left', suffixes=('', '_dup'))

In [18]:
im_final[im_final.duplicated(subset=['Entity--Item'], keep=False)]
# keep first and set columns_with_coverage to nan
im_final = im_final.drop_duplicates(subset=['Entity--Item'], keep='first')
for col in columns_with_coverage:
    if col in im_final.columns:
        im_final[col] = im_final[col].fillna('')

In [19]:
im_final

,Entity--Item,Item Desc 1,Item Desc 2,Qty,Gross Cost,Net Cost,po_cost_amt,Case Pack,VB Flag,VGN,...,Capacity Value 1,Capacity Preferred Metric 1,"Lid, Cover & Cap Type","Lid, Cover & Cap Style",Compatible Product & Product Type,Bottom Diameter (IN),Top Diameter (IN),Product Dimension Type,Wall Height (IN),Product Dimensions
0,1--.20J16C,Cup Foam 20 Oz Tall Coca Cola,Stock Print,43.0,1768,1768,41.070233,500,N,Dart,...,20,Ounce (OZ),,,,2.4,3.7,Tapered & Graduated Product Dimensions,6.1,3.7X6.1X2.4 IN
1,1--.24P,Cup Cold Clr Pp 24 Oz,,28.0,1219,1103,44.790000,600,N,Dart,...,24,Ounce (OZ),,,,2.5,3.9,Tapered & Graduated Product Dimensions,6.1,3.9X6.1X2.5 IN
2,1--.32AJ20C,Cup Foam 32 Oz Coca-Cola,,9.0,645,645,71.720000,400,N,Dart,...,32,Ounce (OZ),,,,2.6,4.2,Tapered & Graduated Product Dimensions,7.1,4.2X7.1X2.6 IN
3,1--.374MS,4 Oz Mistique Design Paper,Hot Cup,40.0,1677,1677,41.590000,1000,N,Dart,...,4,Ounce (OZ),,,,1.8,2.5,Tapered & Graduated Product Dimensions,2.4,2.5X2.4X1.8 IN
4,1--.5C,Cup Cold Clr Pet 5 Oz,Solo Ultra Use Lid L7N25,102.0,13368,12798,127.164608,2500,N,Dart,...,,,,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4488,4--ZIGGI-SO32AC-5,Ziggis V 32Oz Pet Cold Cup Clr 500,,2060.0,129304,129304,62.430000,500,N,Dart,...,,,,,,,,,,
4489,4--ZIGGI-SO374T,Ziggis 4Oz Ppr Hot Cup 20/50,,384.0,14082,14082,37.250000,1000,N,Dart,...,,,,,,,,,,
4490,4--ZIGGI-SOTD24-5,Ziggis V 24Oz Pet Cold Cup Clr 600,,1860.0,74978,74978,39.990000,600,N,Dart,...,,,,,,,,,,
4491,4--ZIGGI-SOTP16D-5,Ziggis V 16Oz Pet Cold Cup Clr 1M,,1365.0,75957,75957,55.870000,1000,N,Dart,...,,,,,,,,,,


#### Write files - this is the subsection of to_keep that fits the category

In [20]:
OP_PATH = f"{parent_dir}\\Data\\{CATEGORY}\\Output\\"
os.makedirs(OP_PATH, exist_ok=True)  
im_final.to_csv(f"{OP_PATH}{CATEGORY}_SKUS_with_Salsify.csv", index=False)

In [21]:
im_final = pd.read_csv(f"{OP_PATH}{CATEGORY}_SKUS_with_Salsify.csv")

## Classification

#### Generate parts for taxonomy prompt

In [22]:
# generate a string of the top 5 values for each column in columns_for_description to help with the prompt
prompt_options_string = cf.get_top_values(im_final, columns_for_description, 5)
prompt_options_string

'Beverage Cup Type: Cup, Lid, Stancap, Cup, Lid & Straw Combo, Cup & Lid Combo \n\n Product Type Collapse: Cup, Lid, Cup Sleeve, Cup Carrier Without Tray, Cup Carrier with Tray \n\n Pack Size: 1000/Case, 500/Case, 600/Case, 2500/Case, 2000/Case \n\n Color: Clear, White, Translucent, Multicolor, Kraft \n\n Material: Polyethylene Terephthalate (PET), Plastic, Paper, Polystyrene (PS), Polylactic Acid (PLA) \n\n Foodservice Global Attributes: Disposable, Disposable|Reusable, Freezer Safe, Dishwasher Safe|Freezer Safe|Microwave Oven Safe, Reusable \n\n Usage Temperature: Cold Only, Hot Only, Cold & Hot \n\n Beverage Cup Style: Insulated, Tumbler, Mug, Souvenir, Wrapped \n\n Sustainable Products: Yes \n\n Product Capacity: 16 OZ, 12 OZ, 8 OZ, 10 OZ, 20 OZ \n\n Capacity Value 1: 16, 12, 8, 10, 20 \n\n Capacity Preferred Metric 1: Ounce (OZ), Fluid Ounce (FLOZ) \n\n Lid, Cover & Cap Type: Flat, Dome, Flat|Vented, Flat|Not Vented, Vented \n\n Lid, Cover & Cap Style: With Hole, No Hole, Sip Thro

In [23]:
# create a description string for each row in that will be used in the promp
im_final['All Descriptions'] = (
    im_final['Item Desc 1'].fillna('') + ' ' +
    im_final['Item Desc 2'].fillna('') + ' ' #+
    #im_final['description_line3_txt'].fillna('')
    ).str.strip()

existing_columns_for_description = [col for col in columns_for_description if col in im_final.columns]

if not existing_columns_for_description:
    print("Warning: None of the specified columns for description exist in the DataFrame.")
    im_final['description'] = "" # Create an empty description column
else:
    if len(existing_columns_for_description) < len(columns_for_description):
        missing_cols = set(columns_for_description) - set(existing_columns_for_description)
        print(f"Warning: The following specified columns were not found in the DataFrame and will be skipped: {missing_cols}")
    
im_final['description'] = im_final.apply(
    lambda row: cf.create_description_string(row, existing_columns_for_description),
    axis=1 # Apply function row-wise
)

In [24]:
output_str = cf.get_most_common_values(prompt_options_string)

In [25]:
output_str

'Beverage Cup Type: Cup | Product Type Collapse: Cup | Pack Size: 1000/Case | Color: Clear | Material: Polyethylene Terephthalate (PET) | Foodservice Global Attributes: Disposable | Usage Temperature: Cold Only | Beverage Cup Style: Insulated | Sustainable Products: Yes | Product Capacity: 16 OZ | Capacity Value 1: 16 | Capacity Preferred Metric 1: Ounce (OZ) | Lid, Cover & Cap Type: Flat | Lid, Cover & Cap Style: With Hole | Compatible Product & Product Type: Cup | Bottom Diameter (IN): 2.4 | Top Diameter (IN): 3.5 | Product Dimension Type: Tapered & Graduated Product Dimensions | Wall Height (IN): 4.0 | Product Dimensions: 3.1X4X1.9 IN'

In [ ]:
example_desc, example_output = cf.explain_top_qty_description(im_final, output_str)

Running cost $0.0000: 100%|██████████| 1/1 [00:02<00:00,  2.71s/chunk]


#### Taxonomy prompts

In [333]:
model = 'gpt-4o-mini' 
chunk_size = 32       
agent = OpenAIAgent(model=model, chunk_size=chunk_size)

In [385]:
im_final = cf.attribute_with_ai(im_final, agent, columns_for_description, prompt_options_string, example_desc, example_output, default_pack_size=1000)

Consider submitting unique prompts to the API to save on costs and time


Running cost $2.5799:   0%|          | 0/141 [00:00<?, ?chunk/s]


KeyboardInterrupt: 

#### Write file with just id and taxonomy

In [386]:
columns_for_description

['Beverage Cup Type',
 'Product Type Collapse',
 'Pack Size',
 'Color',
 'Material',
 'Foodservice Global Attributes',
 'Usage Temperature',
 'Beverage Cup Style',
 'Sustainable Products',
 'Product Capacity',
 'Capacity Value 1',
 'Capacity Preferred Metric 1',
 'Lid, Cover & Cap Type',
 'Lid, Cover & Cap Style',
 'Compatible Product & Product Type',
 'Bottom Diameter (IN)',
 'Top Diameter (IN)',
 'Product Dimension Type',
 'Wall Height (IN)',
 'Product Dimensions']

In [388]:
im_final_attributed, extract_attributes_to_dataframe = cf.extract_attributes_to_dataframe(im_final, columns_for_description, output_excel_filepath=f"{OP_PATH}{CATEGORY}_taxonomy.xlsx", vendor_col = 'VGN')

Processing 4482 rows from the input DataFrame...
Successfully created DataFrame with 4482 rows and 23 columns.


Successfully wrote DataFrame to Excel: c:\Users\zwayne\OneDrive - Advent International\Documents\GitHub\ImperialDadeCategoryManagement\Data\Cups\Output\Cups_taxonomy.xlsx


In [394]:
cf.coverage_improvement(sfy, im_final, extract_attributes_to_dataframe, columns_for_description)

Computing coverage improvement analysis...
Computed post-LLM coverage for 23 columns
Computed initial coverage for 21 columns
Coverage improvement analysis complete. Found 20 columns to compare.


,% Coverage Post LLM,% Initial Coverage,Difference
Column_Name,,,
Beverage Cup Type,57.32%,23.25%,34.07%
Product Type Collapse,55.87%,25.99%,29.88%
Color,56.58%,16.93%,39.65%
Material,62.12%,19.34%,42.77%
Foodservice Global Attributes,18.14%,14.70%,3.44%
Usage Temperature,36.03%,12.03%,24.01%
Beverage Cup Style,11.27%,4.91%,6.36%
Sustainable Products,8.37%,4.44%,3.93%
Product Capacity,85.79%,17.25%,68.54%


#### write file with attributes

In [395]:
im_final_attributed.to_csv(f"{OP_PATH}{CATEGORY}_Attributed.csv", index=False)